In [1]:
from datetime import datetime
from pathlib import Path
import re

import pandas as pd

from utils import write_csv
from models import (
    AssessmentReport,
    DataSource,
    GasType,
    EmissionsCO2e,
    EmissionsTotalCO2e,
    EmissionsTotalSector,
    AggregationType,
)

from utils import display_excel_sheets

from functools import reduce
import pandas as pd

# easier than making custom getter from iso codes
stem_to_actor = {
    "queensland": "AU-QLD",
    "tasmania": "AU-TAS",
    "northern-territory": "AU-NT",
    "victoria": "AU-VIC",
    "western-australia": "AU-WA",
    "act" : "AU-ACT",
    "new-south-wales": "AU-NSW",
    "south-australia": "AU-SA",
}

In [9]:
# ------------------------------------------
# DataSource table
# ------------------------------------------
# https://unfccc.int/topics/mitigation/resources/registry-and-data/ghg-data-from-unfccc
datasource_name = "abs-national-state-and-territory-population"
publisher = "Australian-Bureau-of-Statistics"

datasource = DataSource(
    id=f"{publisher}:{datasource_name}",
    name="National, state and territory population",
    publisher=publisher,
    published_date=datetime.strptime("2025-10-28", "%Y-%m-%d"),
    version="20251028",
    url=f"https://www.abs.gov.au/statistics/people/population/national-state-and-territory-population/latest-release#data-downloads",
)

files = (Path().cwd().parent.parent / "data" / "au-state-populations" / "raw").glob("*.xlsx")

list_of_dfs = []

for fl in files:
    actor_id = stem_to_actor.get(fl.stem)
    sheets = [x for x in display_excel_sheets(fl) if 'data' in x.lower()]

    df_tmp = (
        reduce(lambda l, r: pd.merge(l, r, on="Unnamed: 0", how="outer"),
            [pd.read_excel(fl, sheet_name=s) for s in sheets])
        .rename(columns={'Unnamed: 0': 'date'})
        .assign(date=lambda d: pd.to_datetime(d['date'], errors='coerce'))
        .dropna(subset=['date'])
        .pipe(lambda d: d.loc[:, ['date'] + [c for c in d.columns if 'Persons' in c]])
        .pipe(lambda d: d.assign(
            **{c: pd.to_numeric(d[c], errors='coerce') for c in d.columns if 'Persons' in c}
        ))
        .set_index("date")
        .sum(axis=1, numeric_only=True)
        .rename("population")
        .reset_index(name="population") 
        .assign(
            year=lambda d: d['date'].dt.year,
            datasource_id = datasource.id,
            actor_id = actor_id,
            id = lambda d: d.apply(lambda x: f"{x['actor_id']}:{x['year']}", axis=1),
            )
        .loc[:, ["id", "actor_id", "year", "population", "datasource_id"]]
    )

    list_of_dfs.append(df_tmp)


df = pd.concat(list_of_dfs).astype({"actor_id": str, "year": int, "population": int})

In [10]:
df.sample(20)

,id,actor_id,year,population,datasource_id
23,AU-SA:1994,AU-SA,1994,1463089,Australian-Bureau-of-Statistics:abs-national-s...
8,AU-ACT:1979,AU-ACT,1979,220797,Australian-Bureau-of-Statistics:abs-national-s...
39,AU-SA:2010,AU-SA,2010,1627322,Australian-Bureau-of-Statistics:abs-national-s...
30,AU-QLD:2001,AU-QLD,2001,3571469,Australian-Bureau-of-Statistics:abs-national-s...
42,AU-TAS:2013,AU-TAS,2013,512231,Australian-Bureau-of-Statistics:abs-national-s...
23,AU-NSW:1994,AU-NSW,1994,6044819,Australian-Bureau-of-Statistics:abs-national-s...
5,AU-ACT:1976,AU-ACT,1976,207740,Australian-Bureau-of-Statistics:abs-national-s...
12,AU-NSW:1983,AU-NSW,1983,5352959,Australian-Bureau-of-Statistics:abs-national-s...
29,AU-NSW:2000,AU-NSW,2000,6446558,Australian-Bureau-of-Statistics:abs-national-s...
9,AU-ACT:1980,AU-ACT,1980,224291,Australian-Bureau-of-Statistics:abs-national-s...


,id,actor_id,year,population,datasource_id
0,AU-QLD:1971,AU-QLD,1971,1851485,Australian-Bureau-of-Statistics:abs-national-s...
1,AU-QLD:1972,AU-QLD,1972,1898478,Australian-Bureau-of-Statistics:abs-national-s...
2,AU-QLD:1973,AU-QLD,1973,1951951,Australian-Bureau-of-Statistics:abs-national-s...
3,AU-QLD:1974,AU-QLD,1974,2008340,Australian-Bureau-of-Statistics:abs-national-s...
4,AU-QLD:1975,AU-QLD,1975,2051362,Australian-Bureau-of-Statistics:abs-national-s...
...,...,...,...,...,...
49,AU-SA:2020,AU-SA,2020,1790355,Australian-Bureau-of-Statistics:abs-national-s...
50,AU-SA:2021,AU-SA,2021,1802601,Australian-Bureau-of-Statistics:abs-national-s...
51,AU-SA:2022,AU-SA,2022,1824969,Australian-Bureau-of-Statistics:abs-national-s...
52,AU-SA:2023,AU-SA,2023,1857646,Australian-Bureau-of-Statistics:abs-national-s...
